# TEST SINGLE AUGMENTATION

In [ ]:
from pathlib import Path
import numpy as np
import torch
from torchvision.transforms import ToTensor
import io
from PIL import Image
import cv2
import open3d as o3d

from emma import EmmaDataset, EmmaSearcher, SearchQuery

from models.model import GLPDepth
from types import SimpleNamespace
from collections import OrderedDict

In [ ]:
# Check CUDA Availability

if torch.cuda.is_available():
    print("Ready")
    device = torch.device('cuda')
else:
    print("Not Ready")
    device = torch.device('cpu')

## > DATASET

In [ ]:
dataset_root = None  # e.g. Path("/path/to/EMMa")

if dataset_root is None:
    raise ValueError("Please set `dataset_root` to the path of the EMMa dataset.")

dataset_root = Path(dataset_root)

if not dataset_root.exists():
    raise FileNotFoundError(f"Dataset path does not exist: {dataset_root}")

In [ ]:
# INIT DATASET
ds = EmmaDataset(dataset_root)

In [ ]:
# QUERY
se = EmmaSearcher(ds)
query = SearchQuery(splits=["Test"], require_listed=["mass", "category"], require_present=["package_dimensions"], quantity_range=[1,1], limit=20)
asins = se.query_asins(query)
print(len(asins))

In [ ]:
# SELECT A SAMPLE
asin = asins[4]
rec = ds.get_record(asin)
title = ds.get_title(rec=rec)
categories = ds.get_category_names(rec=rec)

image_dict = ds.get_images(rec)
first_img_bytes = next(iter(image_dict.values()))
first_img = Image.open(io.BytesIO(first_img_bytes)).convert("RGB")
first_img_np = np.array(first_img)

package_dimensions = ds.get_package_dimensions(rec=rec)
dimensions = np.squeeze(package_dimensions) * 2.54 / 100.0
normalization = np.float32(np.linalg.norm(dimensions))

sample = {
    "image": first_img_np,
    "normalization": normalization,
}

print("ASIN: ", asin)
print("Title: ", title)
print("Category Path: ", categories)
print("Package Dimensions: ", package_dimensions)
first_img.show()

## > MODEL

In [ ]:
glp_ckpt = None  # e.g. Path("/path/to/GLP-CKPT")

if glp_ckpt is None:
    raise ValueError("Please set `glp_ckpt` to the path of the GLP-Depth checkpoint.")

glp_ckpt = Path(glp_ckpt)

if not glp_ckpt.exists():
    raise FileNotFoundError(f"Checkpoint path does not exist: {glp_ckpt}")

In [ ]:
# Model Arguments
args = SimpleNamespace(min_depth_eval=1e-3, max_depth_eval=2.63, max_depth=2.63)

In [ ]:
model_weight = torch.load(glp_ckpt)
if 'module' in next(iter(model_weight.items()))[0]:
    model_weight = OrderedDict((k[7:], v) for k, v in model_weight.items())

model = GLPDepth(max_depth=args.max_depth, is_train=False).to(device)
model.load_state_dict(model_weight)
model.eval()

## > AUGMENTATION

In [ ]:
def clamp_pred(args, pred):
    pred = torch.nan_to_num(pred, nan=args.min_depth_eval, posinf=args.max_depth_eval, neginf=args.max_depth_eval)
    return pred.clamp_min(0.0)

def resize_image(image):
    border_size = (640 - 480) // 2
    border_color = [255, 255, 255]
    image = cv2.resize(image, (480, 480))
    image = cv2.copyMakeBorder(image, 0, 0, border_size, border_size, cv2.BORDER_CONSTANT, value=border_color)

    return image

In [ ]:
to_tensor = ToTensor()
input_RGB = to_tensor(resize_image(sample['image'])).unsqueeze(0).to(device)
normalization = sample['normalization']

with torch.no_grad():
    pred = model(input_RGB)
pred_d = pred['pred_d'].squeeze() * normalization
pred_d = clamp_pred(args, pred_d)

# Equivalent saving format to Cardoso
pred_d_img = (pred_d.cpu().numpy() * 1000.0).astype(np.uint16)

## > VISUALIZE

In [ ]:
pred_d_vis = pred_d_img
pred_d_vis = (pred_d_vis - pred_d_vis.min()) / (pred_d_vis.max() - pred_d_vis.min() + 1e-8)
pred_d_vis = (pred_d_vis * 255).astype(np.uint8)

print("DEPTH MIN: ", pred_d.min())
print("DEPTH MAX: ", pred_d.max())
print("NORMALIZATION: ", normalization)
Image.fromarray(pred_d_vis).show()

## > POINT CLOUD

In [ ]:
def pointcloud_from_depth(depth, intrinsics, image=None):
    depth = np.asarray(depth, dtype=np.float32)
    H, W = depth.shape
    K = np.asarray(intrinsics, dtype=np.float32)

    if not np.isfinite(K).all() or K[0, 0] <= 0 or K[1, 1] <= 0:
        raise ValueError("Camera intrinsics must be finite with positive focal lengths.")

    v, u = np.indices((H, W))
    Z = depth
    X = (u - K[0, 2]) * Z / K[0, 0]
    Y = (v - K[1, 2]) * Z / K[1, 1]

    valid = np.isfinite(Z) & (Z > 0)
    points = np.stack((X, Y, Z), axis=-1)[valid].astype(np.float32)

    colors = None
    if image is not None:
        image = np.asarray(image)
        if image.shape[:2] != (H, W):
            image = cv2.resize(image, (W, H), interpolation=cv2.INTER_LINEAR)
        colors = image[..., :3][valid].astype(np.uint8)
    return points, colors

def export_point_cloud_open3d(points, colors, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if len(points) == 0:
        raise ValueError("No valid depth pixels to export.")
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points.astype(np.float64))
    if colors is not None:
        pcd.colors = o3d.utility.Vector3dVector(colors.astype(np.float64) / 255.0)
    if not o3d.io.write_point_cloud(str(out_path), pcd):
        raise IOError(f"Failed to write point cloud: {out_path}")
    print(f"Saved {len(points):,} points to {out_path}")
    return pcd

In [ ]:
fx, fy = None, None  # Set according to legacy vs corrected GLPDepth model

if fx is None or fy is None:
    raise ValueError(
        "Please set `fx` and `fy` according to the "
        "legacy vs corrected GLPDepth model."
    )

fx, fy = float(fx), float(fy)

if fx <= 0 or fy <= 0:
    raise ValueError(
        f"Focal lengths must be positive. Got fx={fx}, fy={fy}."
    )

In [ ]:
depth = pred_d.detach().cpu().numpy().astype(np.float32)
rgb = resize_image(sample["image"])

In [ ]:
camera_height, camera_width = depth.shape
cx, cy = camera_width / 2.0, camera_height / 2.0

K = np.array([
    [fx, 0, cx],
    [0, fy, cy],
    [0, 0, 1],
], dtype=np.float32)

In [ ]:
points, colors = pointcloud_from_depth(depth, K, rgb)

In [ ]:
export_point_cloud_open3d(points, colors, "generated_pointclouds/glpdepth_finetuned.ply")